# Project 6: Test-Driven Code-Repair Agent

Compare a one-shot patch with a three-attempt repair loop using AST mapping,
failure localization, constrained unified diffs, public and hidden tests, and
exact rollback. Gold corrected programs never enter planner context.

Upgrade v1: generated code requires Docker isolation. Hidden tests are used only
once after public-test candidate selection. Historical scores are not new results.
Build the executor on a Docker-enabled host: `docker build -f tools/Dockerfile.sandbox
-t project6-test-runner:p0-20261117 .`. Standard Colab may not provide this runtime.

In [ ]:
from pathlib import Path
import os, runpy, sys
if sys.prefix == sys.base_prefix:
    raise RuntimeError("Use the isolated notebook kernel from SECURITY_UPGRADE.md; shared Colab packages are not modified")
candidates = [Path.cwd(), Path.cwd()/"project6", Path("/content/ai_agentic_attemptings/project6")]
PROJECT_ROOT = next((p.resolve() for p in candidates if (p/"src/project6_agent").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the cloned project directory")
check = runpy.run_path(str(PROJECT_ROOT.parent/"tools/notebook_environment.py"))["validate"]
check(PROJECT_ROOT.parent/"requirements-notebook-cpu.hashes.txt")
os.chdir(PROJECT_ROOT)
source_root = PROJECT_ROOT/"src"
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))
print("Isolated Project 6 CPU/API environment verified; GPU mode requires a separately approved GPU lock")

In [2]:
import getpass,os,sys
from project6_agent.config import load_config
EVAL_BACKEND="local_gpu"  # openai | local_gpu
RUN_FULL_EVAL=True; RUN_API_EVAL=EVAL_BACKEND=="openai"; RUN_LOCAL_GPU_EVAL=EVAL_BACKEND=="local_gpu"
config=load_config(PROJECT_ROOT/"config/default.json")
if RUN_API_EVAL and not os.getenv("OPENAI_API_KEY"):
    if "google.colab" in sys.modules:
        from google.colab import userdata
        key=userdata.get("OPENAI_API_KEY")
    else: key=getpass.getpass("OPENAI_API_KEY (hidden): ")
    if not key: raise RuntimeError("OPENAI_API_KEY required for API mode")
    os.environ["OPENAI_API_KEY"]=key
if RUN_LOCAL_GPU_EVAL:
    import gc,torch
    if not torch.cuda.is_available(): raise RuntimeError("Select a Colab GPU runtime for local_gpu mode")
    stale_names=("source_planner","schema_retriever","local_backend","planners","planner","embedding_retriever","local_answerer")
    released=[name for name in stale_names if globals().pop(name,None) is not None]
    gc.collect(); torch.cuda.empty_cache(); free_bytes,total_bytes=torch.cuda.mem_get_info()
    print({"released_gpu_objects":released,"free_gpu_gib":round(free_bytes/2**30,2),"total_gpu_gib":round(total_bytes/2**30,2)})
print(config.model_dump())

{'released_gpu_objects': [], 'free_gpu_gib': 39.08, 'total_gpu_gib': 39.49}
{'project_id': 'project6', 'seed': 20260802, 'model': 'gpt-5.6-sol', 'reasoning_effort': 'medium', 'max_patch_attempts': 3, 'max_changed_lines': 30, 'test_timeout_seconds': 20.0, 'max_test_output_chars': 12000, 'max_model_calls': 220, 'max_output_tokens': 900, 'max_retries': 2, 'max_estimated_cost_usd': 8.0, 'input_price_per_million_usd': 5.0, 'output_price_per_million_usd': 30.0, 'local_model': 'Qwen/Qwen2.5-Coder-7B-Instruct', 'local_device': 'cuda', 'local_max_new_tokens': 900, 'development_case_count': 4, 'test_case_count': 8}


In [3]:
import subprocess,sys
subprocess.run([sys.executable,"tools/fetch_quixbugs.py"],check=True)
QUIXBUGS_ROOT=PROJECT_ROOT/"data/cache/QuixBugs"

In [4]:
import json
manifest=json.loads((PROJECT_ROOT/"data/quixbugs_manifest.json").read_text())
development_cases=[c for c in manifest["cases"] if c["split"]=="development"]
test_cases=[c for c in manifest["cases"] if c["split"]=="test"]
assert (len(development_cases),len(test_cases))==(4,8)
print({"commit":manifest["commit"],"development":4,"test":8})

{'commit': '4257f44b0ff1181dedaedee6a447e133219fcebf', 'development': 4, 'test': 8}


In [5]:
print("One-shot fixture behavior is covered by tests/test_agent.py; run P06-C07 before API patches.")

One-shot fixture behavior is covered by tests/test_agent.py; run P06-C07 before API patches.


In [6]:
from project6_agent.repository import build_repository_map
symbols=build_repository_map(QUIXBUGS_ROOT/"python_programs")
print({"mapped_symbols":len(symbols),"example":symbols[0].model_dump()})

{'mapped_symbols': 52, 'example': {'path': 'bitcount.py', 'name': 'bitcount', 'kind': 'function', 'line': 2, 'imports': [], 'docstring': None}}


In [7]:
import os,subprocess,sys
test_env=os.environ.copy(); test_env["PYTEST_DISABLE_PLUGIN_AUTOLOAD"]="1"
result=subprocess.run([sys.executable,"-m","pytest","-q","tests"],text=True,capture_output=True,env=test_env,timeout=120); print(result.stdout)
if result.returncode: print(result.stderr); raise RuntimeError("Project 6 tests failed")

.........                                                                [100%]
9 passed in 2.69s



In [ ]:
from runpy import run_path
RunManifest = run_path(str(PROJECT_ROOT.parent / 'tools/run_manifest.py'))['RunManifest']
from project6_agent.agent import RepairAgent
from project6_agent.evaluation import summarize_results
from project6_agent.planners import LocalPatchPlanner, OpenAIPatchPlanner, PatchCallBudget, TransformersPatchBackend
from project6_agent.structured_edits import OpenAIStructuredEditPlanner
from project6_agent.quixbugs import prepare_case
from project6_agent.runner import sandbox_preflight
from datetime import datetime, timezone
import json
if not RUN_FULL_EVAL:
    print("Set RUN_FULL_EVAL=True and RUN_API_EVAL=True after P06-C07 passes.")
else:
    if config.execution_backend != "docker": raise RuntimeError("Model-generated patches require Docker, not trusted_fixture mode")
    if RUN_LOCAL_GPU_EVAL and config.edit_format != "unified_diff":
        raise RuntimeError("Structured edits currently require API mode; no untested local-model adapter")
    sandbox = sandbox_preflight(config.sandbox_image)
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    result_dir=PROJECT_ROOT/"output/runs/upgrade-v2"/EVAL_BACKEND/run_id
    with RunManifest(PROJECT_ROOT, result_dir, config.model_dump()) as run_manifest:
        (result_dir/"run_config.json").write_text(json.dumps({"config": config.model_dump(), "sandbox": sandbox}, indent=2))
        results=[]; planners=[]; agent=RepairAgent(config); runtime=result_dir/"runtime/quixbugs"; runtime.mkdir(parents=True,exist_ok=True)
        local_backend=TransformersPatchBackend(config) if RUN_LOCAL_GPU_EVAL else None
        api_budget = PatchCallBudget(config) if RUN_API_EVAL else None
        for case in test_cases:
            prepared=prepare_case(QUIXBUGS_ROOT,case,runtime/f"source-{case['bug_id']}")
            for system in ["one_shot","repair_loop"]:
                if RUN_LOCAL_GPU_EVAL:
                    planner=LocalPatchPlanner(config,prepared["root"]/prepared["source"],case["bug_id"],prepared["source"],local_backend)
                else:
                    planner_class = OpenAIStructuredEditPlanner if config.edit_format == "structured_edits" else OpenAIPatchPlanner
                    planner=planner_class(config,prepared["root"]/prepared["source"],case["bug_id"],allowed_path=prepared["source"],budget=api_budget)
                planners.append(planner)
                results.append(agent.run(case["bug_id"],system,prepared["root"],runtime/f"work-{case['bug_id']}-{system}",[prepared["source"]],prepared["public_tests"],prepared["hidden_tests"],planner))
        usage=local_backend.usage_summary() if RUN_LOCAL_GPU_EVAL else {"model_calls":sum(p.calls for p in planners),"input_tokens":sum(p.input_tokens for p in planners),"output_tokens":sum(p.output_tokens for p in planners),"estimated_cost_usd":round(sum(p.estimated_cost_usd for p in planners),6)}
        selected_model=config.local_model if RUN_LOCAL_GPU_EVAL else config.model
        if api_budget is not None: usage["reserved_unreported_cost_usd"] = api_budget.reserved
        summary=summarize_results(results,result_dir,planner_usage=usage,model=selected_model,evaluation_backend=EVAL_BACKEND); print(summary)
        run_manifest.record(summary, {name: getattr(globals().get(name), 'model_revision', None) for name in ('source_planner', 'local_backend', 'embedding_retriever')})

In [ ]:
import json
if "result_dir" not in globals(): raise RuntimeError("Run P06-C08 first; historical results are not reused")
path=result_dir/"project6_representative_samples.json"
print(json.loads(path.read_text()) if path.exists() else "Run the paired evaluator first.")

In [ ]:
import subprocess,sys
if "result_dir" not in globals(): raise RuntimeError("Run P06-C08 first; historical results are not reused")
summary_path=result_dir/"project6_final_summary.json"
if summary_path.exists():
    subprocess.run([sys.executable,"tools/generate_report.py","--summary",str(summary_path),"--output",str(result_dir/"project6_report.pdf")],check=True)
    subprocess.run([sys.executable,"tools/validate_project.py"]+([] if RUN_LOCAL_GPU_EVAL else ["--require-results"]),check=True)
else: print("Measured summary absent; report generation skipped.")

## Development-Only Edit Interface Comparison

This optional local/Docker experiment compares model-written unified diffs with
exact source replacements on the four development bugs, under the same three-attempt
budget. The harness validates unique, non-overlapping anchors, Python syntax, paths,
and changed-line limits before generating a diff. It does not retry using final-test
feedback or overwrite the eight-bug test results. Run setup/config/data/tests first;
the test-set evaluation cell is not required. This is API-only, not a GPU workload.

In [ ]:
RUN_EDIT_FORMAT_DEVELOPMENT = False
if RUN_EDIT_FORMAT_DEVELOPMENT:
    import os
    from datetime import datetime, timezone
    from openai import OpenAI
    from project6_agent.edit_comparison import compare_development_edits
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Use the hidden API-key prompt in the configuration cell; never paste a key into source")
    development_output = PROJECT_ROOT/"output/runs/upgrade-v2/development_edits"/datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    development_config = config.model_copy(update={"max_output_tokens": 2048, "reasoning_effort": "low", "max_retries": 0})
    with OpenAI(max_retries=0, timeout=60) as api:
        development_summary = compare_development_edits(api, development_config, PROJECT_ROOT, development_output)
    print(development_summary)
else:
    print("Development-only comparison not requested; no API calls.")

## Real-Repository Benchmark Preparation (v2)

The optional SWE-bench Lite slice contains four upstream development tasks and
eight test tasks. Only issue text, repository and base revision enter planner
context; patches and evaluator tests stay in ignored evaluator-only files. See
`BENCHMARKS.md`. Preparation is not a model or official-harness score.

In [ ]:
from project6_agent.swebench import PUBLIC_FIELDS
import json
public_tasks_path = PROJECT_ROOT / "data/cache/swebench-v2/test_public.json"
if public_tasks_path.exists():
    real_repository_tasks = json.loads(public_tasks_path.read_text())
    assert all(set(task) == set(PUBLIC_FIELDS) for task in real_repository_tasks)
    print({"prepared_test_tasks": len(real_repository_tasks), "model_calls": 0,
           "repositories": sorted({task["repo"] for task in real_repository_tasks})})
else:
    print("Prepare the pinned SWE-bench input using BENCHMARKS.md; no download or model call was started.")